<a href="https://colab.research.google.com/github/mandara-samanalee/FYP_Team_Intellicore/blob/main/notebooks/module3_activity/V2_Module3_Scores_for_Fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Temporal Anomaly Detection: Scoring & Validation - V2

This notebook processes reviews and features to score them based on temporal anomalies detected using an Isolation Forest model. It normalizes anomaly scores and aligns them back with individual user reviews.

### Key Functionality
* **Model Scoring:** Runs a pre-trained Isolation Forest model over sliding temporal window features to evaluate temporal burstiness and anomaly indicators.
* **Normalization:** Normalizes the decision function output to a `[0, 1]` probability-like range using Min-Max scaling, fitted exclusively on the validation set to prevent leakage.
* **Timestamp Alignment:** Maps raw review timestamps to corresponding sliding window intervals to assign precise temporal anomaly scores and outlier flags to each review.

### Inputs
1. **Validation & Test Features:** `features_val_test.csv` (contains sliding window metadata and features for the validation and test sets)
2. **Trained Isolation Forest Model:** `isolation_forest_module3_v2.pkl` (serialized model artifact)
3. **Feature Column List:** `module3_feature_cols_v2.pkl` (serialized list of feature columns used during training)
4. **Review Dataset:** `df_combined_split.csv` (contains individual review details, including product identifiers and timestamps)

### Outputs
* **Final Fusion-Ready Dataset:** `module3_scores_for_fusion.csv` (unified CSV containing aligned scores ready for downstream multi-modal fusion)

### Key Notes
* Scores are normalized such that a higher `temporal_score` represents a more suspicious burst pattern (values closer to 1).
* Real-time interpolation resolves boundaries when a review's timestamp matches multiple active temporal windows by selecting the one closest to the window center.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np

BASE  = "/content/drive/MyDrive"
FINAL = f"{BASE}/Module3_Final"
EXP   = f"{BASE}/module3_burst_experiment"
import joblib

# features (already scored? re-score to be safe)
feats = pd.read_csv(f"{FINAL}/features_val_test.csv")
model = joblib.load(f"{EXP}/isolation_forest_module3_v2.pkl")
fcols = joblib.load(f"{EXP}/module3_feature_cols_v2.pkl")
feats['anomaly_score'] = model.decision_function(feats[fcols])

# normalize: fit on VAL only, apply to both. higher = more suspicious, 0..1
fv = feats[feats['split']=='val']
lo, hi = fv['anomaly_score'].min(), fv['anomaly_score'].max()
feats['temporal_score'] = ((hi - feats['anomaly_score']) / (hi - lo)).clip(0, 1)
feats['flag'] = (feats['anomaly_score'] < -0.07).astype(int)

# window bounds to unix seconds for matching
feats['ws'] = pd.to_datetime(feats['window_start']).astype('int64')//10**9
feats['we'] = pd.to_datetime(feats['window_end']).astype('int64')//10**9
feats['center'] = (feats['ws'] + feats['we'])/2

print(f"Normalization (val-fit): lo={lo:.4f} hi={hi:.4f}")
print(f"temporal_score range: {feats['temporal_score'].min():.3f}–{feats['temporal_score'].max():.3f}")

Mounted at /content/drive
Normalization (val-fit): lo=-0.1350 hi=0.2352
temporal_score range: 0.000–1.000


In [ ]:
df = pd.read_csv(f"{FINAL}/df_combined_split.csv")
reviews = df[df['split'].isin(['val','test'])].copy()

# per-product window lookup
by_prod = {pid: g[['ws','we','center','temporal_score','flag']].values
           for pid, g in feats.groupby('product_id')}

scores, flags = [], []
for pid, ts in zip(reviews['product_id'].values, reviews['timestamp'].values):
    w = by_prod.get(pid)
    if w is None:
        scores.append(0.0); flags.append(0); continue
    cand = w[(w[:,0] <= ts) & (w[:,1] > ts)]      # windows containing timestamp
    if len(cand) == 0:
        scores.append(0.0); flags.append(0)        # fallback: no burst evidence
    else:
        best = cand[np.argmin(np.abs(cand[:,2] - ts))]   # center-nearest
        scores.append(float(best[3])); flags.append(int(best[4]))

reviews['temporal_score'] = scores
reviews['temporal_flag']  = flags
print(f"Scored {len(reviews):,} reviews")
print(f"  flagged: {reviews['temporal_flag'].sum():,}")
print(f"  score>0: {(reviews['temporal_score']>0).sum():,}")

Scored 147,049 reviews
  flagged: 22,071
  score>0: 107,988


In [ ]:
out_cols = ['reviewer_id','product_id','timestamp','temporal_score',
            'temporal_flag','split','is_generated']
handoff = reviews[out_cols]
handoff.to_csv(f"{FINAL}/module3_scores_for_fusion.csv", index=False)

print(f"Saved module3_scores_for_fusion.csv — {len(handoff):,} rows")
print(f"  val: {(handoff['split']=='val').sum():,}, test: {(handoff['split']=='test').sum():,}")
print(handoff.head())

Saved module3_scores_for_fusion.csv — 147,049 rows
  val: 72,804, test: 74,245
                         reviewer_id  product_id   timestamp  temporal_score  \
324446  AHF4SD2F3STTETPKV4AFMSJEPNOQ  B0BW4PFM58  1635695149        0.991024   
324447  AHDPUEUAMXXB6IDSHPLGRBVZUSLA  B0BQRNFHCV  1635695254        0.599160   
324448  AGD62U24GIGKIXNXEWHEISUIU4RA  B0B8MRGWJ5  1635696101        0.000000   
324449  AFUZG4S57VOZIENQ5LJHBPEO5T2Q  B06WV5YJ6H  1635696108        0.000000   
324450  AGHFMCPJDBD7SLMTGXZIZOCWYR5A  B08J83TR22  1635696841        0.796907   

        temporal_flag split  is_generated  
324446              1   val             0  
324447              0   val             0  
324448              0   val             0  
324449              0   val             0  
324450              0   val             0  
